<a href="https://colab.research.google.com/github/rafaelespinosacastaneda/sdl-workshops/blob/main/LAST_VERSION_OT2_BayesianOpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Physical OT-2 Bayesian Optimization with a Gaussian Process

This notebook keeps the physical OT-2 call used in `A.ipynb`:

```python
result, killed = submit_experiment(team_id, R, Y, B)
```

The Bayesian optimization workflow is:

1. Measure the target spectrum.
2. Run **one ordinary random feasible experiment** (not Sobol).
3. Fit a Gaussian Process to the physical experiments observed so far.
4. Generate a pool of feasible 300 µL mixtures.
5. Compute Expected Improvement (EI).
6. Run the highest-EI candidate on the physical OT-2.
7. Add the returned sensor RMSE to the GP training data and repeat.

There is no simulator in this notebook.


In [1]:
%pip install -q gradio_client statsmodels plotly

In [2]:
from gradio_client import Client
import json
import numpy as np
import pandas as pd

from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

# =============================================================================
# Configuration
# =============================================================================
SEED = 999

# Total number of BO experiments, INCLUDING the one random initialization.
NUM_EXPERIMENTS = 3

# Number of candidate mixtures considered when maximizing EI.
N_CANDIDATES = 10000

# Expected-improvement exploration parameter.
XI = 0.01

TOTAL_VOLUME = 300.0
MIN_VOLUME = 20.0

# Feasibility checks for the BO search space.
if MIN_VOLUME < 0:
    raise ValueError("MIN_VOLUME cannot be negative.")

if 3 * MIN_VOLUME > TOTAL_VOLUME:
    raise ValueError(
        "Impossible constraints: 3 * MIN_VOLUME exceeds TOTAL_VOLUME."
    )

CHANNELS = [
    "ch410", "ch440", "ch470", "ch510",
    "ch550", "ch583", "ch620", "ch670",
]

# Replace this with your actual team/student ID.
team_id = ""

# Exact physical OT-2 endpoint used in A.ipynb.
client = Client("https://accelerationconsortium-ot-2-lcm.hf.space/")


def submit_experiment(student_id, r_vol, y_vol, b_vol):
    """
    Submit one experiment to the physical OT-2 service.

    This is intentionally the same calling pattern used in A.ipynb.
    """
    killed = ""

    api_endpoint = "/debug" if student_id == "debug" else "/submit"

    job = client.submit(
        student_id,
        float(r_vol),
        float(y_vol),
        float(b_vol),
        api_name=api_endpoint,
    )

    result = job.result()

    if result["Message"] == "Out of tips. Please contact adminstrator to restock.":
        killed = "tips"
    elif result["Message"] == "Plate full. Please contact adminstrator to replace.":
        killed = "plate"
    elif result["Message"] == "Queue killed. Please contact adminstrator to restart program.":
        killed = "queue"

    return result, killed


def stop_if_killed(killed):
    if killed == "tips":
        raise SystemExit(
            "Queue terminated by administrator. REASON: Out of tips"
        )
    elif killed == "plate":
        raise SystemExit(
            "Queue terminated by administrator. REASON: Plate full"
        )
    elif killed == "queue":
        raise SystemExit(
            "Queue terminated by administrator. REASON: Queue killed"
        )


def extract_sensor_values(result):
    """Return the eight sensor channels as a NumPy vector."""
    if "Sensor Data" not in result:
        raise KeyError(
            "The OT-2 response does not contain 'Sensor Data'. "
            f"Full response: {result}"
        )

    sensor_data = result["Sensor Data"]

    missing = [ch for ch in CHANNELS if ch not in sensor_data]
    if missing:
        raise KeyError(
            f"Missing sensor channels {missing}. Full sensor data: {sensor_data}"
        )

    return np.asarray([sensor_data[ch] for ch in CHANNELS], dtype=float)


def spectral_rmse(measured_values, target_values):
    """RMSE between the measured spectrum and target spectrum."""
    measured_values = np.asarray(measured_values, dtype=float)
    target_values = np.asarray(target_values, dtype=float)

    return float(
        np.sqrt(np.mean((measured_values - target_values) ** 2))
    )


print("Setup complete.")


Loaded as API: https://accelerationconsortium-ot-2-lcm.hf.space/
Setup complete.


In [3]:
# =============================================================================
# Target spectrum
# =============================================================================

TARGET_R = 115.0
TARGET_Y = 163.0
TARGET_B = 22.0

print(
    f"Getting target spectrum for "
    f"R={TARGET_R}, Y={TARGET_Y}, B={TARGET_B}..."
)

# This follows A.ipynb exactly: the target is obtained through submit_experiment.
# This target measurement is NOT counted as the one random BO initialization.
target_result, killed = submit_experiment(
    team_id,
    TARGET_R,
    TARGET_Y,
    TARGET_B,
)

stop_if_killed(killed)

print(json.dumps(target_result, indent=2))

target_values = extract_sensor_values(target_result)

print("\\nTarget sensor values:")
print(target_values)


Getting target spectrum for R=115.0, Y=163.0, B=22.0...
{
  "Status": "Complete",
  "Message": "Experiment completed successfully!",
  "Student ID": "test1",
  "Command": {
    "R": 115.0,
    "Y": 163.0,
    "B": 22.0,
    "well": "B11"
  },
  "Sensor Data": {
    "ch583": 10200,
    "ch670": 20500,
    "ch510": 8432,
    "ch410": 1503,
    "ch620": 12240,
    "ch470": 6752,
    "ch550": 10777,
    "ch440": 8366
  },
  "Experiment ID": "117c2082"
}
\nTarget sensor values:
[ 1503.  8366.  6752.  8432. 10777. 10200. 12240. 20500.]


In [4]:
# =============================================================================
# Gaussian Process + Expected Improvement
# =============================================================================

def normalize_volumes(X):
    """
    Normalize R, Y, B from [0, 300] to [0, 1].

    Every feasible candidate satisfies R + Y + B = 300.
    """
    X = np.asarray(X, dtype=float)
    return X / TOTAL_VOLUME


def fit_gaussian_process(X, y):
    """
    Fit the Gaussian Process surrogate to PHYSICALLY MEASURED experiments.

    X shape: (n_experiments, 3), columns = [R, Y, B]
    y shape: (n_experiments,), objective = measured spectral RMSE
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    X_scaled = normalize_volumes(X)

    # Matern is a standard BO kernel and is more flexible than a pure RBF.
    # WhiteKernel permits a small amount of experimental/sensor noise.
    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=np.ones(3),
            length_scale_bounds=(1e-2, 10.0),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-5,
            noise_level_bounds=(1e-8, 1e0),
        )
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=5 if len(y) >= 2 else 0,
        random_state=SEED,
    )

    gp.fit(X_scaled, y)

    return gp


def expected_improvement(candidates, gp, y_best, xi=0.01):
    """
    Expected Improvement for a MINIMIZATION problem.

    Improvement = y_best - predicted_mean - xi

    Large EI means the candidate has a useful combination of:
    - predicted lower RMSE
    - predictive uncertainty
    """
    candidates = np.asarray(candidates, dtype=float)

    mu, sigma = gp.predict(
        normalize_volumes(candidates),
        return_std=True,
    )

    improvement = y_best - mu - xi

    ei = np.zeros_like(mu, dtype=float)

    nonzero = sigma > 1e-12

    z = np.zeros_like(mu, dtype=float)
    z[nonzero] = improvement[nonzero] / sigma[nonzero]

    ei[nonzero] = (
        improvement[nonzero] * norm.cdf(z[nonzero])
        + sigma[nonzero] * norm.pdf(z[nonzero])
    )

    return ei, mu, sigma


def sample_random_mixture(rng):
    """
    Draw ONE genuinely random feasible mixture.

    Constraints:
        R >= MIN_VOLUME
        Y >= MIN_VOLUME
        B >= MIN_VOLUME
        R + Y + B = TOTAL_VOLUME

    With MIN_VOLUME=20 and TOTAL_VOLUME=300, 20 uL is first assigned
    to each color and the remaining 240 uL is distributed uniformly
    over the 3-component simplex using Dirichlet(1,1,1).

    This is the first BO experiment. It is NOT Sobol.
    """
    remaining_volume = TOTAL_VOLUME - 3 * MIN_VOLUME

    fractions = rng.dirichlet(np.ones(3))
    mixture = MIN_VOLUME + remaining_volume * fractions

    # Floating-point safety: force the total to be exactly TOTAL_VOLUME.
    mixture[2] = TOTAL_VOLUME - mixture[0] - mixture[1]

    return mixture

def make_candidate_pool(rng, n_candidates):
    """
    Generate feasible random candidate mixtures for acquisition optimization.

    Every candidate satisfies:
        R >= MIN_VOLUME
        Y >= MIN_VOLUME
        B >= MIN_VOLUME
        R + Y + B = TOTAL_VOLUME
    """
    remaining_volume = TOTAL_VOLUME - 3 * MIN_VOLUME

    fractions = rng.dirichlet(
        np.ones(3),
        size=n_candidates,
    )

    candidates = MIN_VOLUME + remaining_volume * fractions

    # Floating-point safety: force each row to total exactly TOTAL_VOLUME.
    candidates[:, 2] = (
        TOTAL_VOLUME
        - candidates[:, 0]
        - candidates[:, 1]
    )

    return candidates

def candidate_key(x, decimals=6):
    """Stable key used to avoid intentionally repeating an evaluated candidate."""
    x = np.asarray(x, dtype=float)
    return tuple(np.round(x, decimals=decimals))


def run_physical_experiment(x, target_values):
    """
    Execute one candidate on the physical OT-2 and calculate its sensor RMSE.
    """
    x = np.asarray(x, dtype=float)

    if x.shape != (3,):
        raise ValueError("x must contain exactly [R, Y, B].")

    if not np.all(np.isfinite(x)):
        raise ValueError(f"Candidate contains NaN or infinite values: {x}")

    if np.any(x < MIN_VOLUME - 1e-9):
        raise ValueError(
            f"All BO color volumes must be >= {MIN_VOLUME:.1f} uL. "
            f"Proposed candidate: {x}"
        )

    if not np.isclose(
        x.sum(),
        TOTAL_VOLUME,
        atol=1e-6,
        rtol=0.0,
    ):
        raise ValueError(
            f"Candidate total volume is {x.sum():.6f} µL, "
            f"not {TOTAL_VOLUME:.1f} µL."
        )

    R, Y, B = x

    print(
        f"Submitting to OT-2: "
        f"R={R:.3f}, Y={Y:.3f}, B={B:.3f} µL"
    )

    result, killed = submit_experiment(
        student_id=team_id,
        r_vol=R,
        y_vol=Y,
        b_vol=B,
    )

    stop_if_killed(killed)

    print(json.dumps(result, indent=2))

    measured_values = extract_sensor_values(result)
    rmse = spectral_rmse(measured_values, target_values)

    return measured_values, rmse, result


print("Gaussian Process and acquisition functions defined.")


Gaussian Process and acquisition functions defined.


In [5]:
# =============================================================================
# Dry-run validation of the BO search space (NO robot call)
# =============================================================================

_validation_rng = np.random.default_rng(SEED)

_test_first = sample_random_mixture(_validation_rng)
_test_pool = make_candidate_pool(_validation_rng, 10000)

assert np.all(_test_first >= MIN_VOLUME - 1e-9)
assert np.isclose(_test_first.sum(), TOTAL_VOLUME, atol=1e-6, rtol=0.0)

assert np.all(_test_pool >= MIN_VOLUME - 1e-9)
assert np.all(
    np.isclose(
        _test_pool.sum(axis=1),
        TOTAL_VOLUME,
        atol=1e-6,
        rtol=0.0,
    )
)

print("Search-space validation passed.")
print("First random sample:", _test_first)
print("First random sample total:", _test_first.sum())
print("Minimum volume in candidate pool:", _test_pool.min())


Search-space validation passed.
First random sample: [165.69235544  55.1875826   79.12006196]
First random sample total: 300.0
Minimum volume in candidate pool: 20.00260410851476


In [6]:
# =============================================================================
# Bayesian Optimization
#
# Experiment 1: one plain random feasible sample
# Experiments 2+: GP + Expected Improvement
# Every objective evaluation is returned by the physical OT-2.
# =============================================================================

if NUM_EXPERIMENTS < 1:
    raise ValueError("NUM_EXPERIMENTS must be at least 1.")

rng = np.random.default_rng(SEED)

X = []
y = []
records = []
tested = set()


# -----------------------------------------------------------------------------
# 1. ONE RANDOM INITIAL EXPERIMENT
# -----------------------------------------------------------------------------
x_random = sample_random_mixture(rng)

# Defensive validation before the physical call.
if np.any(x_random < MIN_VOLUME - 1e-9):
    raise RuntimeError(f"Random initialization violated minimum volume: {x_random}")
if not np.isclose(x_random.sum(), TOTAL_VOLUME, atol=1e-6, rtol=0.0):
    raise RuntimeError(f"Random initialization does not sum to {TOTAL_VOLUME}: {x_random}")

print(
    f"\\n{'=' * 70}\\n"
    f"EXPERIMENT 1/{NUM_EXPERIMENTS}: RANDOM INITIALIZATION\\n"
    f"{'=' * 70}"
)

measured_values, rmse, result = run_physical_experiment(
    x_random,
    target_values,
)

X.append(x_random.copy())
y.append(rmse)
tested.add(candidate_key(x_random))

records.append(
    {
        "iteration": 1,
        "selection": "random",
        "R": x_random[0],
        "Y": x_random[1],
        "B": x_random[2],
        "total_volume": x_random.sum(),
        "rmse": rmse,
    }
)

print(f"Measured RMSE = {rmse:.6f}")


# -----------------------------------------------------------------------------
# 2. BAYESIAN OPTIMIZATION FROM EXPERIMENT 2 ONWARD
# -----------------------------------------------------------------------------
for iteration in range(1, NUM_EXPERIMENTS):

    X_array = np.asarray(X, dtype=float)
    y_array = np.asarray(y, dtype=float)

    # STEP 1: fit GP to ALL physical experiments observed so far.
    gp = fit_gaussian_process(
        X_array,
        y_array,
    )

    # STEP 2: generate feasible candidates.
    candidates = make_candidate_pool(
        rng,
        N_CANDIDATES,
    )

    # Do not deliberately submit an already-tested point.
    keep = np.array(
        [candidate_key(x) not in tested for x in candidates],
        dtype=bool,
    )

    candidates = candidates[keep]

    if len(candidates) == 0:
        raise RuntimeError(
            "Candidate pool contained no untested candidates. "
            "Increase N_CANDIDATES."
        )

    # STEP 3: calculate Expected Improvement.
    y_best = float(np.min(y_array))

    ei, mu, sigma = expected_improvement(
        candidates,
        gp,
        y_best,
        xi=XI,
    )

    # STEP 4: choose the candidate with the maximum EI.
    best_idx = int(np.argmax(ei))
    x_next = candidates[best_idx].copy()

    # Defensive validation before the physical call.
    if np.any(x_next < MIN_VOLUME - 1e-9):
        raise RuntimeError(f"EI candidate violated minimum volume: {x_next}")
    if not np.isclose(x_next.sum(), TOTAL_VOLUME, atol=1e-6, rtol=0.0):
        raise RuntimeError(f"EI candidate does not sum to {TOTAL_VOLUME}: {x_next}")

    predicted_mu = float(mu[best_idx])
    predicted_sigma = float(sigma[best_idx])
    selected_ei = float(ei[best_idx])

    print(
        f"\\n{'=' * 70}\\n"
        f"EXPERIMENT {iteration + 1}/{NUM_EXPERIMENTS}: "
        f"GP EXPECTED IMPROVEMENT\\n"
        f"{'=' * 70}"
    )

    print(
        "Selected by GP/EI:\\n"
        f"  R = {x_next[0]:.3f} µL\\n"
        f"  Y = {x_next[1]:.3f} µL\\n"
        f"  B = {x_next[2]:.3f} µL\\n"
        f"  GP predicted RMSE = {predicted_mu:.6f}\\n"
        f"  GP uncertainty    = {predicted_sigma:.6f}\\n"
        f"  Expected Improvement = {selected_ei:.6f}"
    )

    # STEP 5: PHYSICALLY RUN THE GP-SELECTED CANDIDATE.
    measured_values, rmse, result = run_physical_experiment(
        x_next,
        target_values,
    )

    # STEP 6: append THE SAME POINT that was actually sent to the robot.
    X.append(x_next.copy())
    y.append(rmse)
    tested.add(candidate_key(x_next))

    records.append(
        {
            "iteration": iteration + 1,
            "selection": "GP + Expected Improvement",
            "R": x_next[0],
            "Y": x_next[1],
            "B": x_next[2],
            "total_volume": x_next.sum(),
            "rmse": rmse,
            "gp_predicted_rmse_before_measurement": predicted_mu,
            "gp_sigma_before_measurement": predicted_sigma,
            "expected_improvement": selected_ei,
        }
    )

    print(f"Physical measured RMSE = {rmse:.6f}")
    print(f"Best measured RMSE so far = {np.min(y):.6f}")


# =============================================================================
# Results
# =============================================================================
results_df = pd.DataFrame(records)
results_df["best_so_far"] = results_df["rmse"].cummin()

results_df["feasible"] = (
    (results_df[["R", "Y", "B"]] >= MIN_VOLUME - 1e-9).all(axis=1)
    & np.isclose(
        results_df["total_volume"],
        TOTAL_VOLUME,
        atol=1e-6,
        rtol=0.0,
    )
)

display(results_df)

best_idx = int(np.argmin(y))
best_x = np.asarray(X[best_idx], dtype=float)
best_rmse = float(y[best_idx])

print("\\nBest PHYSICALLY MEASURED experiment:")
print(f"  R = {best_x[0]:.3f} µL")
print(f"  Y = {best_x[1]:.3f} µL")
print(f"  B = {best_x[2]:.3f} µL")
print(f"  RMSE = {best_rmse:.6f}")


\n======================================================================\nEXPERIMENT 1/3: RANDOM INITIALIZATION\n======================================================================
Submitting to OT-2: R=165.692, Y=55.188, B=79.120 µL
{
  "Status": "Complete",
  "Message": "Experiment completed successfully!",
  "Student ID": "test1",
  "Command": {
    "R": 165.69235544375863,
    "Y": 55.1875826001157,
    "B": 79.12006195612568,
    "well": "C2"
  },
  "Sensor Data": {
    "ch583": 7333,
    "ch670": 14412,
    "ch510": 6724,
    "ch410": 1161,
    "ch620": 9041,
    "ch470": 5405,
    "ch550": 8217,
    "ch440": 5994
  },
  "Experiment ID": "39c016f9"
}
Measured RMSE = 3011.346356
\n======================================================================\nEXPERIMENT 2/3: GP EXPECTED IMPROVEMENT\n======================================================================
Selected by GP/EI:\n  R = 20.461 µL\n  Y = 257.988 µL\n  B = 21.551 µL\n  GP predicted RMSE = 3011.346356\n  GP uncert

/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


{
  "Status": "Complete",
  "Message": "Experiment completed successfully!",
  "Student ID": "test1",
  "Command": {
    "R": 20.46078263321065,
    "Y": 257.98793363450557,
    "B": 21.551283732283764,
    "well": "C3"
  },
  "Sensor Data": {
    "ch583": 7995,
    "ch670": 19995,
    "ch510": 9626,
    "ch410": 1341,
    "ch620": 10127,
    "ch470": 6358,
    "ch550": 11010,
    "ch440": 6418
  },
  "Experiment ID": "c5d48a16"
}
Physical measured RMSE = 1371.036469
Best measured RMSE so far = 1371.036469
\n======================================================================\nEXPERIMENT 3/3: GP EXPECTED IMPROVEMENT\n======================================================================
Selected by GP/EI:\n  R = 23.428 µL\n  Y = 252.912 µL\n  B = 23.660 µL\n  GP predicted RMSE = 1633.804099\n  GP uncertainty    = 601.642350\n  Expected Improvement = 131.168466
Submitting to OT-2: R=23.428, Y=252.912, B=23.660 µL
{
  "Status": "Complete",
  "Message": "Experiment completed successfull

,iteration,selection,R,Y,B,total_volume,rmse,gp_predicted_rmse_before_measurement,gp_sigma_before_measurement,expected_improvement,best_so_far,feasible
0,1,random,165.692355,55.187583,79.120062,300.0,3011.346356,NaN,NaN,NaN,3011.346356,True
1,2,GP + Expected Improvement,20.460783,257.987934,21.551284,300.0,1371.036469,3011.346356,0.025022,0.005769,1371.036469,True
2,3,GP + Expected Improvement,23.428114,252.911707,23.660179,300.0,1137.638069,1633.804099,601.642350,131.168466,1137.638069,True


\nBest PHYSICALLY MEASURED experiment:
  R = 23.428 µL
  Y = 252.912 µL
  B = 23.660 µL
  RMSE = 1137.638069


In [ ]:
# Optional: save the physical campaign history.
results_df.to_csv("ot2_gp_bayesian_optimization_results.csv", index=False)
print("Saved: ot2_gp_bayesian_optimization_results.csv")
